# Urdu Sign Language Recognition — Complete Implementation
## CS-324 Machine Learning | Batch 2023 — Spring 2026

| Roll Number | Name |
|---|---|
| CS-23066 | Assamir Zafar |
| CS-23111 | Umaima Khan |
| CS-23042 | Umar Saleem |
| CS-23090 | Fasih Ahmed Khan |

**Teacher** Dr. Maria Waqas

---

### Notebook Contents

| Part | Description |
|---|---|
| **Part 1** | Install & Global Imports |
| **Part 2** | Preprocessing — 128×128, 224×224 crops, ANN Landmarks CSV |
| **Part 3** | CNN 1 — EfficientNetB0 (128×128) |
| **Part 4** | CNN 2 — EfficientNetB3 (224×224) |
| **Part 5** | CNN 3 — ResNet50 (224×224) |
| **Part 6** | CNN 4 — MobileNetV2 (128×128) |
| **Part 7** | CNN Comparison — All 4 Models |
| **Part 8** | CNN Winner Variants — EfficientNetB3 Variant A (SGD + StepLR) |
| **Part 9** | CNN Winner Variants — EfficientNetB3 Variant B (RMSprop + Cosine LR) |
| **Part 10** | CNN Winner Variants Comparison |
| **Part 11** | ANN Setup — Feature Engineering & Data Splits |
| **Part 12** | ANN 1 — Strategy A Baseline (XYZ Only) |
| **Part 13** | ANN 2 — Strategy A Improved (XYZ Only) |
| **Part 14** | ANN 3 — Strategy B Baseline (XYZ + Angles) |
| **Part 15** | ANN 4 — Strategy B Improved (XYZ + Angles) ★ Winner |
| **Part 16** | ANN Four-Model Comparison |
| **Part 17** | ANN Winner Variants — Strategy B Improved Variant A (Higher LR + L1L2) |
| **Part 18** | ANN Winner Variants — Strategy B Improved Variant B (Heavier Dropout + Cosine) |
| **Part 19** | ANN Winner Variants Comparison |
| **Part 20** | Overall Project Winner — CNN vs ANN |
| **Part 21** | Random Image Testing — CNN Winner + ANN Winner |

## REFER TO DATASET HERE:
**[USL _DATASET](https://www.kaggle.com/datasets/assamirzafar/usl-dataset)**


---
## Part 1 — Install & Global Imports

All libraries used across the entire notebook are imported once here. MediaPipe is pinned to 0.10.13 for Kaggle P100 compatibility.

In [ ]:
!pip install mediapipe==0.10.13

In [ ]:
import os, cv2, csv, shutil, random, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib; matplotlib.rcParams['figure.dpi'] = 150
import mediapipe as mp
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, models as km
from tensorflow.keras.applications import (EfficientNetB0, EfficientNetB3,
                                            ResNet50, MobileNetV2)
from tensorflow.keras.applications.efficientnet  import preprocess_input as eff_pre
from tensorflow.keras.applications.resnet50       import preprocess_input as rn_pre
from tensorflow.keras.applications.mobilenet_v2   import preprocess_input as mv2_pre
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau,
                                         ModelCheckpoint, CSVLogger,
                                         LearningRateScheduler)
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

CLASS_NAMES = ['Alif','Background','Bari Yay','Bay','Chay','Ddaal',
               'Gaaf','Laam','Say','Suad','Wow','Zay','Zuad','Zzay']
N_CLASSES   = len(CLASS_NAMES)

print('TensorFlow :', tf.__version__)
print('MediaPipe  :', mp.__version__)
print('Classes    :', CLASS_NAMES)

---
## Part 2 — Preprocessing Pipeline

Three outputs are generated from the raw dataset:

| Output | Resolution | Used By |
|---|---|---|
| `USL_Split_128/` | 128×128 | EfficientNetB0, MobileNetV2 |
| `USL_Split_224/` | 224×224 | EfficientNetB3, ResNet50 |
| `landmarks.csv`  | 63 cols  | All ANN models |

### Why MediaPipe Cropping?
MediaPipe detects 21 hand landmarks and extracts a tight bounding box (with 30px padding) around the hand. Cropping removes background clutter and forces each model to learn hand shape only — not the environment. Background class images are resized directly without hand detection.

### Why No Horizontal Flip?
USL signs are directionally specific. A horizontally flipped hand represents a different or non-existent sign. Horizontal flip is disabled in all augmentation pipelines.

### 2.1 — Path Configuration

In [ ]:
RAW_ROOT    = '/kaggle/input/datasets/assamirzafar/usl-dataset/ML CEP Dataset'
RENAMED_DIR = os.path.join(RAW_ROOT, 'Renamed_Dataset')

CNN_128_DIR = '/kaggle/working/CNN_Processed_128'
CNN_224_DIR = '/kaggle/working/CNN_Processed_224'
SPLIT_128   = '/kaggle/working/USL_Split_128'
SPLIT_224   = '/kaggle/working/USL_Split_224'
ANN_CSV     = '/kaggle/working/landmarks.csv'
PADDING     = 30
CONF        = 0.3

SKIP_NAMES  = {'CNN_Processed','ANN_Landmarks','scripts','Renamed_Dataset'}
valid_cls   = sorted([d for d in os.listdir(RENAMED_DIR)
                      if os.path.isdir(os.path.join(RENAMED_DIR,d))
                      and d not in SKIP_NAMES])
print('Classes found:', valid_cls)
print('Count        :', len(valid_cls))

### 2.2 — MediaPipe Crop Function

In [ ]:
mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
IMG_EXT    = ('.jpg','.jpeg','.png','.bmp')

def mediapipe_crop_pipeline(src, out_dir, size, skip_log):
    os.makedirs(out_dir, exist_ok=True)
    total = saved = 0; skipped = []
    with mp_hands.Hands(static_image_mode=True, max_num_hands=1,
                        min_detection_confidence=CONF) as hands:
        for cls in valid_cls:
            cls_in = os.path.join(src,cls)
            cls_out= os.path.join(out_dir,cls)
            os.makedirs(cls_out, exist_ok=True)
            imgs = [f for f in os.listdir(cls_in) if f.lower().endswith(IMG_EXT)]
            for fname in imgs:
                total += 1
                img = cv2.imread(os.path.join(cls_in,fname))
                if img is None: skipped.append(fname); continue
                h,w = img.shape[:2]
                if cls.lower()=='background':
                    crop = cv2.resize(img,(size,size))
                else:
                    res = hands.process(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
                    if not res.multi_hand_landmarks:
                        skipped.append(os.path.join(cls,fname)); continue
                    lms=res.multi_hand_landmarks[0].landmark
                    xs=[l.x*w for l in lms]; ys=[l.y*h for l in lms]
                    x1=max(0,int(min(xs))-PADDING); y1=max(0,int(min(ys))-PADDING)
                    x2=min(w,int(max(xs))+PADDING); y2=min(h,int(max(ys))+PADDING)
                    crop=cv2.resize(img[y1:y2,x1:x2],(size,size))
                cv2.imwrite(os.path.join(cls_out,fname),crop)
                saved += 1
    with open(skip_log,'w') as f:
        f.write(f'Total:{total} Saved:{saved} Skipped:{len(skipped)}\n')
        for p in skipped: f.write(p+'\n')
    print(f'  {size}×{size} → Saved:{saved}/{total} | Skipped:{len(skipped)}')

### 2.3 — Generate Both CNN Crop Resolutions

In [ ]:
print('Generating 128×128 crops...')
mediapipe_crop_pipeline(RENAMED_DIR,CNN_128_DIR,128,'/kaggle/working/skip_128.txt')
print('Generating 224×224 crops...')
mediapipe_crop_pipeline(RENAMED_DIR,CNN_224_DIR,224,'/kaggle/working/skip_224.txt')

### 2.4 — Train / Val / Test Split (Stratified 80 / 10 / 10)

In [ ]:
def split_dataset(src, out):
    for sp in ['train','val','test']:
        for cls in valid_cls:
            os.makedirs(os.path.join(out,sp,cls),exist_ok=True)
    for cls in valid_cls:
        imgs=sorted([f for f in os.listdir(os.path.join(src,cls))
                     if f.lower().endswith(IMG_EXT)])
        random.seed(SEED); random.shuffle(imgs)
        n=len(imgs); nv=max(1,int(n*.10)); nt=max(1,int(n*.10))
        mp={'val':imgs[:nv],'test':imgs[nv:nv+nt],'train':imgs[nv+nt:]}
        for sp,fs in mp.items():
            for f in fs:
                shutil.copy(os.path.join(src,cls,f),os.path.join(out,sp,cls,f))
    for sp in ['train','val','test']:
        n=sum(len(os.listdir(os.path.join(out,sp,c))) for c in valid_cls)
        print(f'  {sp:6s}: {n}')

print('Splitting 128×128:')
split_dataset(CNN_128_DIR,SPLIT_128)
print('Splitting 224×224:')
split_dataset(CNN_224_DIR,SPLIT_224)

### 2.5 — ANN Landmark Extraction → landmarks.csv

Each image produces 63 values: x, y, z coordinates for all 21 MediaPipe hand landmarks, **wrist-normalised** so landmark 0 is always (0,0,0). Background class is excluded — it has no hand to extract landmarks from.

In [ ]:
def generate_landmarks_csv(src,out_csv):
    header=[f'{ax}{i}' for i in range(21) for ax in ['x','y','z']]+['label']
    rows=[]; skipped=0
    with mp_hands.Hands(static_image_mode=True,max_num_hands=1,
                        min_detection_confidence=CONF) as hands:
        for cls in valid_cls:
            if cls.lower()=='background': continue
            cls_dir=os.path.join(src,cls)
            for fname in os.listdir(cls_dir):
                if not fname.lower().endswith(IMG_EXT): continue
                img=cv2.imread(os.path.join(cls_dir,fname))
                if img is None: skipped+=1; continue
                res=hands.process(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
                if not res.multi_hand_landmarks: skipped+=1; continue
                c=[]
                for l in res.multi_hand_landmarks[0].landmark:
                    c.extend([l.x,l.y,l.z])
                c[0::3]=[v-c[0] for v in c[0::3]]
                c[1::3]=[v-c[1] for v in c[1::3]]
                c[2::3]=[v-c[2] for v in c[2::3]]
                rows.append(c+[cls])
    with open(out_csv,'w',newline='') as f:
        w=csv.writer(f); w.writerow(header); w.writerows(rows)
    print(f'landmarks.csv → {len(rows)} rows | {skipped} skipped')

print('Generating landmarks.csv...')
generate_landmarks_csv(RENAMED_DIR,ANN_CSV)

### 2.6 — Class Distribution Plot

In [ ]:
df_lm=pd.read_csv(ANN_CSV); counts=df_lm['label'].value_counts().sort_index()
fig,ax=plt.subplots(figsize=(13,4))
bars=ax.bar(counts.index,counts.values,color='steelblue',edgecolor='white')
ax.bar_label(bars,fmt='%d',padding=3,fontsize=9)
ax.set_title('Class Distribution — USL Dataset',fontsize=13,fontweight='bold')
ax.set_xlabel('Class'); ax.set_ylabel('Count'); ax.set_ylim(0,counts.max()+40)
plt.xticks(rotation=30,ha='right'); plt.tight_layout()
plt.savefig('/kaggle/working/class_distribution.png',dpi=150); plt.show()
print(f'Total landmark rows: {len(df_lm)}')

---
## Shared CNN Utilities

Defined once, reused by all CNN models.

**Focal Loss:** Down-weights easy examples (e.g. Background) so the model focuses on hard similar signs (Gaaf/Zay).
- α=0.25 for EfficientNetB0 (original binary detection default — kept for baseline authenticity)
- α=1.0 for EfficientNetB3 (corrected for 14-class multi-class)
- ResNet50 and MobileNetV2 use standard Categorical Crossentropy.

**Augmentation** is applied at training time only. Val/Test generators apply preprocessing only — no augmentation.

In [ ]:
# ── Focal Loss ────────────────────────────────────────────────────────────────
def focal_loss(gamma=2.0,alpha=0.25):
    def loss(y_true,y_pred):
        y_pred=tf.clip_by_value(y_pred,1e-8,1.0)
        ce=-y_true*tf.math.log(y_pred)
        wt=y_true*tf.math.pow(1-y_pred,gamma)
        return tf.reduce_mean(tf.reduce_sum(alpha*wt*ce,axis=1))
    return loss

# ── Augmentation generators ───────────────────────────────────────────────────
def make_generators(preprocess_fn, train_dir, val_dir, test_dir, size, bs=32):
    aug = ImageDataGenerator(
        preprocessing_function=preprocess_fn,
        rotation_range=15, brightness_range=[0.7,1.3],
        zoom_range=0.1, shear_range=10,
        width_shift_range=0.1, height_shift_range=0.1,
        fill_mode='nearest', horizontal_flip=False)
    plain = ImageDataGenerator(preprocessing_function=preprocess_fn)
    tr  = aug.flow_from_directory(train_dir,target_size=(size,size),
              batch_size=bs,class_mode='categorical',shuffle=True,seed=SEED)
    val = plain.flow_from_directory(val_dir,target_size=(size,size),
              batch_size=bs,class_mode='categorical',shuffle=False)
    te  = plain.flow_from_directory(test_dir,target_size=(size,size),
              batch_size=bs,class_mode='categorical',shuffle=False)
    return tr,val,te

# ── Full CNN Evaluation Function ──────────────────────────────────────────────
def cnn_evaluate(model,test_gen,title,prefix):
    test_gen.reset()
    pp=model.predict(test_gen,verbose=0)
    yp=np.argmax(pp,axis=1); yt=test_gen.classes
    acc=np.mean(yp==yt)
    print(f'\n{"="*55}\n  {title}\n{"="*55}')
    print(f'  Test Accuracy: {acc*100:.2f}%')
    rep=classification_report(yt,yp,target_names=CLASS_NAMES,output_dict=True)
    print(classification_report(yt,yp,target_names=CLASS_NAMES))
    macro_f1=round(rep['macro avg']['f1-score'],4)

    # Confusion Matrix
    cm=confusion_matrix(yt,yp)
    fig,ax=plt.subplots(figsize=(12,10))
    im=ax.imshow(cm,cmap='Blues'); plt.colorbar(im,ax=ax)
    ax.set_xticks(range(N_CLASSES)); ax.set_yticks(range(N_CLASSES))
    ax.set_xticklabels(CLASS_NAMES,rotation=45,ha='right',fontsize=9)
    ax.set_yticklabels(CLASS_NAMES,fontsize=9)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'{title}\nConfusion Matrix — Test Acc: {acc*100:.2f}%',fontweight='bold')
    thresh=cm.max()/2
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=8,
                    color='white' if cm[i,j]>thresh else 'black')
    plt.tight_layout(); plt.savefig(f'/kaggle/working/{prefix}_cm.png',dpi=150); plt.show()

    # Precision / Recall / F1
    prec=[rep[c]['precision'] for c in CLASS_NAMES]
    rec =[rep[c]['recall']    for c in CLASS_NAMES]
    f1  =[rep[c]['f1-score']  for c in CLASS_NAMES]
    x=np.arange(N_CLASSES); w=0.25
    fig,ax=plt.subplots(figsize=(15,5))
    ax.bar(x-w,prec,w,label='Precision',color='steelblue',alpha=0.85)
    ax.bar(x,  rec, w,label='Recall',   color='coral',   alpha=0.85)
    ax.bar(x+w,f1,  w,label='F1',       color='seagreen',alpha=0.85)
    for i,(p,r,f) in enumerate(zip(prec,rec,f1)):
        ax.text(i-w,p+.005,f'{p:.2f}',ha='center',va='bottom',fontsize=6.5)
        ax.text(i,  r+.005,f'{r:.2f}',ha='center',va='bottom',fontsize=6.5)
        ax.text(i+w,f+.005,f'{f:.2f}',ha='center',va='bottom',fontsize=6.5)
    ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES,rotation=30,ha='right')
    ax.set_ylim(0,1.12); ax.legend(); ax.grid(axis='y',alpha=0.3)
    ax.set_title(f'{title} — Precision / Recall / F1',fontweight='bold')
    plt.tight_layout(); plt.savefig(f'/kaggle/working/{prefix}_prf.png',dpi=150); plt.show()

    # ROC-AUC
    yb=label_binarize(yt,classes=list(range(N_CLASSES)))
    fig,ax=plt.subplots(figsize=(10,8))
    cols=plt.cm.tab20(np.linspace(0,1,N_CLASSES)); aucs={}
    for i,(cls,col) in enumerate(zip(CLASS_NAMES,cols)):
        fpr,tpr,_=roc_curve(yb[:,i],pp[:,i])
        a=auc(fpr,tpr); aucs[cls]=round(a,4)
        ax.plot(fpr,tpr,color=col,lw=1.5,label=f'{cls} (AUC={a:.3f})')
    ax.plot([0,1],[0,1],'k--',lw=1,label='Random')
    mean_auc=round(np.mean(list(aucs.values())),4)
    ax.set_title(f'{title}\nROC-AUC (Mean={mean_auc})',fontweight='bold')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.legend(fontsize=7,loc='lower right'); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(f'/kaggle/working/{prefix}_roc.png',dpi=150); plt.show()

    return {'model':title,'test_acc':round(acc*100,2),'mean_auc':mean_auc,'macro_f1':macro_f1}

def plot_cnn_history(hists,labels,title,prefix,ft_epoch=None):
    accs =sum([h.history['accuracy']     for h in hists],[])
    vaccs=sum([h.history['val_accuracy'] for h in hists],[])
    lss  =sum([h.history['loss']         for h in hists],[])
    vlss =sum([h.history['val_loss']     for h in hists],[])
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    fig.suptitle(title,fontweight='bold')
    axes[0].plot(accs, label='Train Acc',color='steelblue')
    axes[0].plot(vaccs,label='Val Acc',  color='orange')
    if ft_epoch: axes[0].axvline(ft_epoch,color='red',linestyle='--',lw=1.5,label='Fine-tune start')
    axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(lss, label='Train Loss',color='steelblue')
    axes[1].plot(vlss,label='Val Loss',  color='orange')
    if ft_epoch: axes[1].axvline(ft_epoch,color='red',linestyle='--',lw=1.5)
    axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(f'/kaggle/working/{prefix}_lc.png',dpi=150); plt.show()

print('CNN helpers ready.')

---
## Part 3 — CNN Model 1: EfficientNetB0 (128×128)

**Baseline CNN.** EfficientNetB0 was selected as the starting point — compact (5.3M params), well-studied, and available with ImageNet pretrained weights. The 128×128 resolution and shallow head define the performance floor.

### Architecture
```
EfficientNetB0 (ImageNet, frozen) → GAP → BatchNorm → Dropout(0.4) → Dense(14, softmax)
```

### Training Strategy
- **Phase A:** Base frozen, head only — 10 epochs, lr=1e-3
- **Phase B:** Last 20 layers unfrozen — 18 epochs, lr=1e-5
- **Phase C:** Last 10 layers — 5 epochs, lr=1e-6

**Loss:** Focal Loss (γ=2.0, α=0.25) | **Optimizer:** Adam

In [ ]:
B0_TR,B0_VAL,B0_TE = make_generators(eff_pre,
    SPLIT_128+'/train',SPLIT_128+'/val',SPLIT_128+'/test',128)

base_b0=EfficientNetB0(weights='imagenet',include_top=False,input_shape=(128,128,3))
base_b0.trainable=False
x=base_b0.output; x=layers.GlobalAveragePooling2D()(x)
x=layers.BatchNormalization()(x); x=layers.Dropout(0.4)(x)
out=layers.Dense(14,activation='softmax')(x)
b0=keras.Model(base_b0.input,out)
b0.compile(optimizer=keras.optimizers.Adam(1e-3),
           loss=focal_loss(2.0,0.25),metrics=['accuracy'])
b0.summary()

In [ ]:
print('Phase A — frozen base')
h_b0a=b0.fit(B0_TR,validation_data=B0_VAL,epochs=10,verbose=1)

for l in base_b0.layers[-20:]: l.trainable=True
b0.compile(optimizer=keras.optimizers.Adam(1e-5),
           loss=focal_loss(2.0,0.25),metrics=['accuracy'])
print('Phase B — last 20 layers')
h_b0b=b0.fit(B0_TR,validation_data=B0_VAL,epochs=18,verbose=1)

for l in base_b0.layers[:-10]: l.trainable=False
b0.compile(optimizer=keras.optimizers.Adam(1e-6),
           loss=focal_loss(2.0,0.25),metrics=['accuracy'])
print('Phase C — last 10 layers')
h_b0c=b0.fit(B0_TR,validation_data=B0_VAL,epochs=5,verbose=1)

b0.save('/kaggle/working/b0_final.keras')
print('B0 saved.')

In [ ]:
plot_cnn_history([h_b0a,h_b0b,h_b0c],['Phase A','Phase B','Phase C'],
                 'EfficientNetB0 — Training History (All Phases)','b0',
                 ft_epoch=len(h_b0a.epoch))
r_b0=cnn_evaluate(b0,B0_TE,'EfficientNetB0 (128×128)','b0')

---
## Part 4 — CNN Model 2: EfficientNetB3 (224×224)

**Improved CNN.** Every limitation of B0 is targeted:

| Limitation | Fix |
|---|---|
| B0 undersized | Upgraded to B3 (12M params, compound scaling) |
| 128×128 too low | Increased to 224×224 (appropriate for B3) |
| Focal α=0.25 wrong | Corrected to α=1.0 for 14-class multi-class |
| Shallow head | Dense(256)→Drop(0.5)→Dense(128)→Drop(0.3) |
| No callbacks | EarlyStopping + ReduceLROnPlateau + ModelCheckpoint |

### Architecture
```
EfficientNetB3 (ImageNet, frozen) → GAP → BN → Dense(256,relu) → Drop(0.5)
                                  → Dense(128,relu) → Drop(0.3) → Dense(14,softmax)
```

In [ ]:
# ============================================================================
# CELL 1: MODEL DEFINITION & HEAD ARCHITECTURE
# ============================================================================
from tensorflow.keras.applications import EfficientNetB3
from tensorflow import keras
from tensorflow.keras import layers

# 1. Data Generator Setup
B3_TR, B3_VAL, B3_TE = make_generators(
    eff_pre,
    SPLIT_224 + '/train',
    SPLIT_224 + '/val',
    SPLIT_224 + '/test',
    224
)

# 2. Base Model Setup (Frozen Feature Extractor)
base_b3 = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_b3.trainable = False

# 3. Dense Head Construction
x = base_b3.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)

# First Dense Block
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)

# Second Dense Block
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)

# 14-Class Output Layer (Huruf-e-tahajji Classifier)
out = layers.Dense(14, activation='softmax')(x)

# 4. Compile Model for Phase A
b3 = keras.Model(base_b3.input, out)
b3.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=focal_loss(2.0, 1.0),
    metrics=['accuracy']
)

b3.summary()

In [ ]:
# ============================================================================
# CELL 2: PROGRESSIVE TRAINING (PHASE A & PHASE B)
# ============================================================================
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# ----------------------------------------------------------------------------
# PHASE A: Feature Extraction (Dense Head Tuning Only)
# ----------------------------------------------------------------------------
cb_b3a = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('/kaggle/working/best_b3_phaseA.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

print("=== Starting B3 Phase A — Base Frozen ===")
h_b3a = b3.fit(B3_TR, validation_data=B3_VAL, epochs=18, callbacks=cb_b3a, verbose=1)


# ----------------------------------------------------------------------------
# PHASE B: Targeted Fine-Tuning (Unfreezing Deep Architectural Layers)
# ----------------------------------------------------------------------------
# Reload best weights from Phase A to prevent gradient shocks
b3.load_weights('/kaggle/working/best_b3_phaseA.keras')

# Unfreeze the final 10 layers of the backbone for fine-tuning
for l in base_b3.layers[-10:]: 
    l.trainable = True

# Recompile with a strictly conservative learning rate (1e-5)
b3.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=focal_loss(2.0, 1.0),
    metrics=['accuracy']
)

cb_b3b = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('/kaggle/working/best_b3_phaseB.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

print("\n=== Starting B3 Phase B — Last 10 Layers Unfrozen ===")
h_b3b = b3.fit(B3_TR, validation_data=B3_VAL, epochs=20, callbacks=cb_b3b, verbose=1)


# ----------------------------------------------------------------------------
# WEIGHT SELECTION & FINAL EXPORT
# ----------------------------------------------------------------------------
bestA = max(h_b3a.history['val_accuracy'])
bestB = max(h_b3b.history['val_accuracy'])

print("\n=== Evaluation Results ===")
if bestA >= bestB:
    print(f"Phase A wins ({bestA:.4f}) — Retaining Phase A checkpoints.")
    b3.load_weights('/kaggle/working/best_b3_phaseA.keras')
else:
    print(f"Phase B wins ({bestB:.4f}) — Retaining fine-tuned features.")
    # No reload needed as b3 currently holds the best-fit Phase B weights natively

# Save the final consolidated deployment asset
b3.save('/kaggle/working/b3_final.keras')
print("b3_final.keras exported successfully.")

In [ ]:
# Phase comparison bar
fig,ax=plt.subplots(figsize=(6,5))
ax.bar(['Phase A\n(Frozen)','Phase B\n(Fine-tuned)'],
       [bestA*100,bestB*100],color=['steelblue','orange'],edgecolor='white')
for i,v in enumerate([bestA*100,bestB*100]):
    ax.text(i,v+.1,f'{v:.2f}%',ha='center',fontweight='bold')
ax.set_ylim(85,100); ax.set_ylabel('Val Accuracy (%)')
ax.set_title('EfficientNetB3 Phase A vs Phase B',fontweight='bold'); ax.grid(axis='y',alpha=0.3)
plt.tight_layout(); plt.savefig('/kaggle/working/b3_phase_cmp.png',dpi=150); plt.show()
plot_cnn_history([h_b3a, h_b3b], ['Phase A', 'Phase B'],
                 'EfficientNetB3 — Phase A & B Training History', 'b3',
                 ft_epoch=len(h_b3a.epoch))
r_b3=cnn_evaluate(b3,B3_TE,'EfficientNetB3 (224×224)','b3')

---
## Part 5 — CNN Model 3: ResNet50 (224×224)

ResNet50 uses **residual (skip) connections** that allow gradients to bypass layers during backpropagation, solving the vanishing gradient problem and enabling training of 50-layer deep networks. It has 25.6M parameters.

### Architecture
```
ResNet50 (ImageNet, frozen) → GAP → Dense(512, relu) → Dropout(0.5) → Dense(14, softmax)
```

### Training Strategy
- **Phase A:** Base frozen, 10 epochs, lr=1e-3, EarlyStopping(patience=3)
- **Phase B:** Last 32 layers unfrozen (layers 143+), 10 epochs, lr=1e-4, EarlyStopping(patience=5)

**Loss:** Categorical Crossentropy | **Optimizer:** Adam

In [ ]:
RN_TR,RN_VAL,RN_TE = make_generators(rn_pre,
    SPLIT_224+'/train',SPLIT_224+'/val',SPLIT_224+'/test',224)

base_rn=ResNet50(weights='imagenet',include_top=False,input_shape=(224,224,3))
base_rn.trainable=False
x=base_rn.output; x=layers.GlobalAveragePooling2D()(x)
x=layers.Dense(512,activation='relu')(x); x=layers.Dropout(0.5)(x)
out=layers.Dense(14,activation='softmax')(x)
rn=keras.Model(base_rn.input,out)
rn.compile(optimizer=keras.optimizers.Adam(1e-3),
           loss='categorical_crossentropy',metrics=['accuracy'])
rn.summary()

In [ ]:
cb_rna=[ModelCheckpoint('/kaggle/working/rn_phaseA.keras',save_best_only=True,
                         monitor='val_accuracy',verbose=1),
        EarlyStopping(patience=3,restore_best_weights=True,verbose=1)]
print('ResNet50 Phase A — frozen')
h_rna=rn.fit(RN_TR,validation_data=RN_VAL,epochs=10,callbacks=cb_rna,verbose=1)

base_rn.trainable=True
for l in base_rn.layers[:143]: l.trainable=False
rn.compile(optimizer=keras.optimizers.Adam(1e-4),
           loss='categorical_crossentropy',metrics=['accuracy'])
cb_rnb=[ModelCheckpoint('/kaggle/working/rn_phaseB.keras',save_best_only=True,
                         monitor='val_accuracy',verbose=1),
        EarlyStopping(patience=5,restore_best_weights=True,verbose=1)]
print('ResNet50 Phase B — last 32 layers')
h_rnb=rn.fit(RN_TR,validation_data=RN_VAL,
             epochs=len(h_rna.epoch)+10,
             initial_epoch=len(h_rna.epoch),
             callbacks=cb_rnb,verbose=1)

rn.save('/kaggle/working/rn_final.keras'); print('ResNet50 saved.')

In [ ]:
plot_cnn_history([h_rna,h_rnb],[],
                 'ResNet50 — Training History','rn',
                 ft_epoch=len(h_rna.epoch))
r_rn=cnn_evaluate(rn,RN_TE,'ResNet50 (224×224)','rn')

### ResNet50 Performance Analysis

Based on the generated training curves, here is the diagnostic breakdown of your ResNet50 implementation:

* **Phase A (Epochs 1 to 10):** The model shows typical behavior for a frozen backbone configuration. Accuracy rises steadily while loss drops smoothly, finding a solid baseline layout without altering the core weights.
* **Phase B (Epochs 10 to 18):** Unfreezing the last 32 layers creates a massive capacity shift. Because the learning rate is conservative ($1\times10^{-4}$), the model doesn't destroy its prior representations. Instead, it successfully pushes validation accuracy past **94%**, which outperforms both EfficientNet architectures.

---

### Overfitting Assessment: Is it Overfitted?

**No, this model is not overfitted.** It is actually highly stable and showing excellent generalization behavior.

Here is why based on your specific graphs:

1. **Parallel Trend Invariance:** Look closely at the Loss History between epochs 12 and 18. Both the training loss (blue) and validation loss (red) are descending smoothly together in parallel. If the model were overfitting, the red validation curve would flatten out or bow upward while the blue curve continued down.
2. **Negligible Generalization Gap:** The variance between final training accuracy and final validation accuracy is extremely tight (under a couple of percentage points). There is no "accuracy stagnation gap" where training hits 99% while validation trails far behind at 91%.
3. **Absence of Validation Jitter:** The validation curves remain exceptionally smooth through the end of Phase B. Massive, parameter-heavy architectures like ResNet50 typically swing wildly up and down between epochs when they start over-indexing on noise. The clean trajectory here proves the regularizers, early stopping patience, and targeted fine-tuning limitations are working perfectly.

---

---
## Part 6 — CNN Model 4: MobileNetV2 (128×128)

MobileNetV2 uses **depthwise separable convolutions** and **inverted residual blocks with linear bottlenecks** — factoring a standard convolution into depthwise (per-channel) and pointwise (1×1) steps. This reduces computation by up to 8-9× while retaining accuracy. Designed for mobile and edge deployment.

### Architecture
```
MobileNetV2 (ImageNet, frozen) → GAP → Dropout(0.3) → Dense(256, relu) → Dense(14, softmax)
```

### Training Strategy
- **Phase A:** Base frozen, 20 epochs, lr=1e-4, EarlyStopping(patience=5)
- **Phase B:** Layers 100+ unfrozen, 10 epochs, lr=1e-5

**Loss:** Categorical Crossentropy | **Optimizer:** Adam

In [ ]:
MV2_TR,MV2_VAL,MV2_TE = make_generators(mv2_pre,
    SPLIT_128+'/train',SPLIT_128+'/val',SPLIT_128+'/test',128)

base_mv2=MobileNetV2(weights='imagenet',include_top=False,input_shape=(128,128,3))
base_mv2.trainable=False
mv2=km.Sequential([
    base_mv2, layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3), layers.Dense(256,activation='relu'),
    layers.Dense(14,activation='softmax')])
mv2.compile(optimizer=keras.optimizers.Adam(1e-4),
            loss='categorical_crossentropy',metrics=['accuracy'])
mv2.summary()

In [ ]:
cb_mv2a=[ModelCheckpoint('/kaggle/working/mv2_phaseA.keras',save_best_only=True,
                          monitor='val_accuracy',verbose=1),
         EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True,verbose=1)]
print('MobileNetV2 Phase A — frozen')
h_mv2a=mv2.fit(MV2_TR,validation_data=MV2_VAL,epochs=20,callbacks=cb_mv2a,verbose=1)

base_mv2.trainable=True
for l in base_mv2.layers[:100]: l.trainable=False
mv2.compile(optimizer=keras.optimizers.Adam(1e-5),
            loss='categorical_crossentropy',metrics=['accuracy'])
print('MobileNetV2 Phase B — layers 100+')
h_mv2b=mv2.fit(MV2_TR,validation_data=MV2_VAL,
               epochs=len(h_mv2a.epoch)+10,
               initial_epoch=h_mv2a.epoch[-1],
               verbose=1)

mv2.save('/kaggle/working/mv2_final.keras'); print('MobileNetV2 saved.')

In [ ]:
plot_cnn_history([h_mv2a,h_mv2b],[],
                 'MobileNetV2 — Training History','mv2',
                 ft_epoch=len(h_mv2a.epoch))
r_mv2=cnn_evaluate(mv2,MV2_TE,'MobileNetV2 (128×128)','mv2')

---
## Part 7 — CNN Four-Model Comparison

All four CNN models are compared on the same held-out test set.

In [ ]:
cnn_results=[r_b0,r_b3,r_rn,r_mv2]
cnn_df=pd.DataFrame(cnn_results)
cnn_df.to_csv('/kaggle/working/cnn_comparison.csv',index=False)
print('\n'+'='*65)
print('  CNN FOUR-MODEL COMPARISON')
print('='*65)
print(cnn_df.to_string(index=False))

labels_c=['EfficientNetB0\n(128×128)','EfficientNetB3\n(224×224)',
          'ResNet50\n(224×224)','MobileNetV2\n(128×128)']
x=np.arange(len(labels_c)); w=0.25
fig,ax=plt.subplots(figsize=(14,6))
b1=ax.bar(x-w,[r['test_acc'] for r in cnn_results],w,label='Test Acc (%)',color='steelblue',alpha=0.9)
b2=ax.bar(x,  [r['mean_auc']*100 for r in cnn_results],w,label='AUC×100',color='orange',alpha=0.9)
b3_b=ax.bar(x+w,[r['macro_f1']*100 for r in cnn_results],w,label='F1×100',color='seagreen',alpha=0.9)
for bars in [b1,b2,b3_b]: ax.bar_label(bars,fmt='%.1f',padding=2,fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels_c,fontsize=10)
ax.set_ylim(0,115); ax.set_title('CNN — Four-Model Comparison',fontsize=13,fontweight='bold')
ax.legend(); ax.grid(axis='y',alpha=0.3)
plt.tight_layout(); plt.savefig('/kaggle/working/cnn_4model_cmp.png',dpi=150); plt.show()

# RESNET 50 2 remaining implimentations


## Part 8 — CNN Winner Variant A: ResNet50 with SGD + Step LR Decay

**Hyperparameter changes from the original ResNet50:**

| Parameter | Original ResNet50 | Variant A |
| --- | --- | --- |
| Optimizer | Adam (lr=1e-3) | **SGD (momentum=0.9, nesterov=True, lr=0.01)** |
| LR Schedule | ReduceLROnPlateau | **Step Decay — halve every 5 epochs** |
| Weight Decay | None | **L2=1e-4 on head Dense layers** |
| Head Dropout | 0.5, 0.3 | 0.5, 0.3 (same) |

**Why SGD?** SGD with Nesterov momentum often generalizes better than Adam on image classification because it finds flatter minima. Step decay provides more predictable LR reduction than plateau-based reduction.

Architecture extends the original ResNet50 head with an additional BatchNormalization layer after GAP and a second Dense(128, relu) block before the output. Loss is also changed from Categorical Crossentropy to Focal Loss (γ=2.0, α=1.0). The table above captures the primary hyperparameter differences; the architectural additions serve as extra regularization for the SGD optimizer.

In [ ]:

# Step decay schedule
def step_decay(epoch, lr):
    if (epoch+1) % 5 == 0:
        return lr * 0.5
    return lr

base_rnv1 = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_rnv1.trainable = False

x = base_rnv1.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(14, activation='softmax')(x)

# FIXED: Correctly passes the ResNet50 input layer to the functional Model
rnv1 = keras.Model(base_rnv1.input, out)

rnv1.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.01, momentum=0.9, nesterov=True),
    loss=focal_loss(2.0, 1.0), metrics=['accuracy'])
rnv1.summary()


In [ ]:
cb_rnv1 = [
    LearningRateScheduler(step_decay, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    ModelCheckpoint('/kaggle/working/rnv1_best.keras',
                    monitor='val_accuracy', save_best_only=True, verbose=1)]

print('RESNET50 Variant A — SGD + Step Decay')
h_rnv1 = rnv1.fit(RN_TR, validation_data=RN_VAL,
                  epochs=25, callbacks=cb_rnv1, verbose=1)

# Phase B fine-tune
for l in base_rnv1.layers[-10:]: 
    l.trainable = True

rnv1.compile(
    optimizer=keras.optimizers.SGD(learning_rate=1e-4, momentum=0.9, nesterov=True),
    loss=focal_loss(2.0, 1.0), metrics=['accuracy'])

# FIXED: Correctly passing RN_TR and referencing h_rnv1 history lengths
h_rnv1b = rnv1.fit(RN_TR, validation_data=RN_VAL,
                   epochs=len(h_rnv1.epoch) + 10,
                   initial_epoch=len(h_rnv1.epoch),
                   verbose=1)

# FIXED: Saves the correct rnv1 model instance instead of the legacy b3v1
rnv1.save('/kaggle/working/RN_variant_a.keras')
print('RESNET50 Variant A saved.')

In [ ]:
# Plot history for ResNet50 Variant A
plot_cnn_history([h_rnv1, h_rnv1b], [],
                 'RESNET50 Variant A (SGD + Step Decay) — Training History', 'rnv1',
                 ft_epoch=len(h_rnv1.epoch))

# Evaluate ResNet50 Variant A on the test set
r_rnv1 = cnn_evaluate(rnv1, RN_TE, 'RESNET50 Variant A (SGD + Step Decay)', 'rnv1')


---

## Part 9 — CNN Winner Variant B: ResNet50 with RMSprop + Cosine LR

**Hyperparameter changes from the original ResNet50:**

| Parameter | Original ResNet50 | Variant B |
| --- | --- | --- |
| Optimizer | Adam (lr=1e-3) | **RMSprop (lr=5e-4, rho=0.9)** |
| LR Schedule | ReduceLROnPlateau | **Cosine Annealing (CosineDecay)** |
| Head Dense 1 | Dense(256) | **Dense(512)** — wider |
| Head Dropout 1 | 0.5 | **0.6** — heavier regularization |

**Why RMSprop + Cosine?** RMSprop adapts per-parameter learning rates by tracking a moving average of squared gradients — often better for noisy image gradients than Adam on small datasets. Cosine annealing decays the LR along a smooth cosine curve to a near-zero minimum, which often results in a better final convergence point than plateau-based decay.---


In [ ]:
import math

# Calculate steps using the ResNet training generator
total_steps_v2 = 25 * len(RN_TR)  # approx steps for cosine schedule
cosine_lr = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=5e-4, decay_steps=total_steps_v2, alpha=1e-6)

# Load ResNet50 backbone
base_rnv2 = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_rnv2.trainable = False

x = base_rnv2.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.6)(x)
x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(14, activation='softmax')(x)

# FIXED: Properly passing the ResNet50 input layer to build the functional model
rnv2 = keras.Model(base_rnv2.input, out)

rnv2.compile(
    optimizer=keras.optimizers.RMSprop(learning_rate=cosine_lr, rho=0.9),
    loss=focal_loss(2.0, 1.0), metrics=['accuracy'])
rnv2.summary()

In [ ]:
cb_rnv2 = [
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    ModelCheckpoint('/kaggle/working/rnv2_best.keras',
                    monitor='val_accuracy', save_best_only=True, verbose=1)]

print('RESNET50 Variant B — RMSprop + Cosine LR')
h_rnv2 = rnv2.fit(RN_TR, validation_data=RN_VAL,
                  epochs=25, callbacks=cb_rnv2, verbose=1)

# Phase B fine-tune
for l in base_rnv2.layers[-10:]: 
    l.trainable = True

cosine_ft = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=5e-6, decay_steps=10 * len(RN_TR), alpha=1e-8)

rnv2.compile(
    optimizer=keras.optimizers.RMSprop(learning_rate=cosine_ft, rho=0.9),
    loss=focal_loss(2.0, 1.0), metrics=['accuracy'])

# FIXED: Passing RN_TR/RN_VAL and tracking h_rnv2 history states
h_rnv2b = rnv2.fit(RN_TR, validation_data=RN_VAL,
                   epochs=len(h_rnv2.epoch) + 10,
                   initial_epoch=len(h_rnv2.epoch),
                   verbose=1)

# FIXED: Saving the correct ResNet model instance
rnv2.save('/kaggle/working/RN_variant_b.keras')
print('RESNET50 Variant B saved.')

In [ ]:
# Plot history for ResNet50 Variant B
plot_cnn_history([h_rnv2, h_rnv2b], [],
                 'RESNET50 Variant B (RMSprop + Cosine LR) — Training History', 'rnv2',
                 ft_epoch=len(h_rnv2.epoch))

# Evaluate ResNet50 Variant B on the test set
r_rnv2 = cnn_evaluate(rnv2, RN_TE, 'RESNET50 Variant B (RMSprop + Cosine)', 'rnv2')

---
## Part 10 — CNN Winner Variants Comparison

The original RESNET50 is compared against its two hyperparameter variants.

In [ ]:
# Collection of ResNet50 variants evaluated dictionaries
rn_variants=[r_rn, r_rnv1, r_rnv2]
rnv_df=pd.DataFrame(rn_variants)
rnv_df.to_csv('/kaggle/working/rn_variants_comparison.csv',index=False)
print(rnv_df.to_string(index=False))

vnames=['Original\n(Adam+ReduceLR)','Variant A\n(SGD+StepDecay)','Variant B\n(RMSprop+Cosine)']
x=np.arange(3); w=0.25
fig,ax=plt.subplots(figsize=(12,5))
b1=ax.bar(x-w,[r['test_acc']    for r in rn_variants],w,label='Test Acc (%)',color='steelblue',alpha=0.9)
b2=ax.bar(x,  [r['mean_auc']*100 for r in rn_variants],w,label='AUC×100',    color='orange',   alpha=0.9)
b3b=ax.bar(x+w,[r['macro_f1']*100 for r in rn_variants],w,label='F1×100',    color='seagreen', alpha=0.9)
for bars in [b1,b2,b3b]: ax.bar_label(bars,fmt='%.1f',padding=2,fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(vnames,fontsize=10)
ax.set_ylim(0,115)
# FIXED: Title string changed to ResNet50
ax.set_title('ResNet50 — Original vs Variant A vs Variant B',fontsize=13,fontweight='bold')
ax.legend(); ax.grid(axis='y',alpha=0.3)
plt.tight_layout(); plt.savefig('/kaggle/working/rn_variants_cmp.png',dpi=150); plt.show()

# Val accuracy overlay
fig,ax=plt.subplots(figsize=(11,4))
# FIXED: Updated history references to use h_rna, h_rnv1, and h_rnv2
ax.plot(h_rna.history['val_accuracy'], label='Original (Adam)',  color='steelblue',lw=2)
ax.plot(h_rnv1.history['val_accuracy'],label='Variant A (SGD)',  color='orange',   lw=2)
ax.plot(h_rnv2.history['val_accuracy'],label='Variant B (RMSprop)',color='seagreen',lw=2)
# FIXED: Title string changed to ResNet50
ax.set_title('ResNet50 Variants — Val Accuracy Overlay (Phase A)',fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val Accuracy')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('/kaggle/working/rn_variants_overlay.png',dpi=150); plt.show()

---
## Part 11 — ANN Setup: Feature Engineering & Data Preparation

### Two Feature Strategies

| Strategy | Input Dim | Features |
|---|---|---|
| **A — XYZ Only** | 63 | Raw wrist-normalised x, y, z for 21 landmarks |
| **B — XYZ + Angles** | 78 | 63 XYZ + 15 derived finger joint angles |

### Finger Joint Angles (Strategy B)
15 angles computed using the dot-product formula at the middle joint of each landmark triplet A→B→C:
```
angle = arccos( (BA · BC) / (|BA| × |BC|) )
```
Angles are **rotation-invariant** — the same gesture produces the same angles regardless of hand position in frame. Triplets cover all finger joints (thumb, index, middle, ring, pinky) plus 4 cross-knuckle angles.

In [ ]:
# Load and deduplicate
df_lm=pd.read_csv(ANN_CSV).drop_duplicates().reset_index(drop=True)
feat_cols=[c for c in df_lm.columns if c!='label']
X_raw=df_lm[feat_cols].values.astype(np.float32)

le=LabelEncoder(); y_int=le.fit_transform(df_lm['label'])
y_cat=to_categorical(y_int)
ANN_CLASSES=list(le.classes_); N_ANN=len(ANN_CLASSES)

with open('/kaggle/working/usl_label_encoder.pkl','wb') as f:
    pickle.dump(le,f)

# Angle features
def angle_between(a,b,c):
    ba=a-b; bc=c-b
    cv=np.einsum('ij,ij->i',ba,bc)/(np.linalg.norm(ba,axis=1)*np.linalg.norm(bc,axis=1)+1e-8)
    return np.arccos(np.clip(cv,-1.,1.))

def extract_angles(X):
    def lm(i): return X[:,i*3:i*3+3]
    T=[(0,1,2),(1,2,3),(2,3,4),(5,6,7),(6,7,8),(9,10,11),(10,11,12),
       (13,14,15),(14,15,16),(17,18,19),(18,19,20),
       (0,5,9),(5,9,13),(9,13,17),(0,17,13)]
    return np.stack([angle_between(lm(a),lm(b),lm(c)) for a,b,c in T],axis=1).astype(np.float32)

X_ang=extract_angles(X_raw)
X_b_raw=np.concatenate([X_raw,X_ang],axis=1)

# Splits + scaling
def make_ann_splits(X,y_cat,y_int):
    Xtr,Xtmp,ytr_c,ytmp_c,ytr_i,ytmp_i=train_test_split(
        X,y_cat,y_int,test_size=0.20,random_state=SEED,stratify=y_int)
    Xval,Xte,yval_c,yte_c,yval_i,yte_i=train_test_split(
        Xtmp,ytmp_c,ytmp_i,test_size=0.50,random_state=SEED,stratify=ytmp_i)
    sc=StandardScaler()
    Xtr=sc.fit_transform(Xtr); Xval=sc.transform(Xval); Xte=sc.transform(Xte)
    return Xtr,Xval,Xte,ytr_c,yval_c,yte_c,ytr_i,yval_i,yte_i,sc

(A_tr,A_val,A_te,A_ytr,A_yval,A_yte,A_ytr_i,A_yval_i,A_yte_i,sc_a)=make_ann_splits(X_raw,y_cat,y_int)
(B_tr,B_val,B_te,B_ytr,B_yval,B_yte,B_ytr_i,B_yval_i,B_yte_i,sc_b)=make_ann_splits(X_b_raw,y_cat,y_int)

cw=compute_class_weight('balanced',classes=np.unique(y_int),y=y_int)
class_weights=dict(enumerate(cw))

print(f'Strategy A: {A_tr.shape[1]} features | Train:{A_tr.shape[0]} Val:{A_val.shape[0]} Test:{A_te.shape[0]}')
print(f'Strategy B: {B_tr.shape[1]} features')
print(f'Classes: {ANN_CLASSES}')

### ANN Shared Evaluation Helper

Defined once — used by all 4 ANN models and all variants. Generates confusion matrix, precision/recall/F1, and ROC-AUC for every model.

In [ ]:
def ann_evaluate(model,Xtr,Xval,Xte,ytr_c,yval_c,yte_c,ytr_i,yval_i,yte_i,hist,title,prefix):
    ytr_p  = np.argmax(model.predict(Xtr, verbose=0),axis=1)
    yval_p = np.argmax(model.predict(Xval,verbose=0),axis=1)
    pp     = model.predict(Xte,verbose=0)
    yte_p  = np.argmax(pp,axis=1)
    tr_acc = np.mean(ytr_p==ytr_i)
    va_acc = np.mean(yval_p==yval_i)
    te_acc = np.mean(yte_p==yte_i)
    print(f'\n{"="*55}\n  {title}\n{"="*55}')
    print(f'  Train:{tr_acc*100:.2f}%  Val:{va_acc*100:.2f}%  Test:{te_acc*100:.2f}%')
    rep=classification_report(yte_i,yte_p,target_names=ANN_CLASSES,output_dict=True)
    print(classification_report(yte_i,yte_p,target_names=ANN_CLASSES))
    macro_f1=round(rep['macro avg']['f1-score'],4)

    # Learning curves
    fig,axes=plt.subplots(1,2,figsize=(13,4))
    fig.suptitle(f'{title} — Training History',fontweight='bold')
    axes[0].plot(hist.history['accuracy'],    label='Train Acc',color='steelblue')
    axes[0].plot(hist.history['val_accuracy'],label='Val Acc',  color='orange')
    axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(hist.history['loss'],    label='Train Loss',color='steelblue')
    axes[1].plot(hist.history['val_loss'],label='Val Loss',  color='orange')
    axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(f'/kaggle/working/{prefix}_lc.png',dpi=150); plt.show()

    # Confusion Matrix
    cm=confusion_matrix(yte_i,yte_p)
    fig,ax=plt.subplots(figsize=(12,10))
    im=ax.imshow(cm,cmap='Blues'); plt.colorbar(im,ax=ax)
    ax.set_xticks(range(N_ANN)); ax.set_yticks(range(N_ANN))
    ax.set_xticklabels(ANN_CLASSES,rotation=45,ha='right',fontsize=9)
    ax.set_yticklabels(ANN_CLASSES,fontsize=9)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'{title}\nConfusion Matrix — Test:{te_acc*100:.2f}%',fontweight='bold')
    thresh=cm.max()/2
    for i in range(N_ANN):
        for j in range(N_ANN):
            ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=8,
                    color='white' if cm[i,j]>thresh else 'black')
    plt.tight_layout(); plt.savefig(f'/kaggle/working/{prefix}_cm.png',dpi=150); plt.show()

    # PRF
    prec=[rep[c]['precision'] for c in ANN_CLASSES]
    rec =[rep[c]['recall']    for c in ANN_CLASSES]
    f1  =[rep[c]['f1-score']  for c in ANN_CLASSES]
    x=np.arange(N_ANN); w2=0.25
    fig,ax=plt.subplots(figsize=(15,5))
    ax.bar(x-w2,prec,w2,label='Precision',color='steelblue',alpha=0.85)
    ax.bar(x,   rec, w2,label='Recall',   color='coral',   alpha=0.85)
    ax.bar(x+w2,f1,  w2,label='F1',       color='seagreen',alpha=0.85)
    for i,(p,r,f) in enumerate(zip(prec,rec,f1)):
        ax.text(i-w2,p+.005,f'{p:.2f}',ha='center',va='bottom',fontsize=6.5)
        ax.text(i,   r+.005,f'{r:.2f}',ha='center',va='bottom',fontsize=6.5)
        ax.text(i+w2,f+.005,f'{f:.2f}',ha='center',va='bottom',fontsize=6.5)
    ax.set_xticks(x); ax.set_xticklabels(ANN_CLASSES,rotation=30,ha='right')
    ax.set_ylim(0,1.12); ax.legend(); ax.grid(axis='y',alpha=0.3)
    ax.set_title(f'{title} — Precision / Recall / F1',fontweight='bold')
    plt.tight_layout(); plt.savefig(f'/kaggle/working/{prefix}_prf.png',dpi=150); plt.show()

    # ROC-AUC
    fig,ax=plt.subplots(figsize=(10,8)); cols=plt.cm.tab20(np.linspace(0,1,N_ANN)); aucs={}
    for i,(cls,col) in enumerate(zip(ANN_CLASSES,cols)):
        fpr,tpr,_=roc_curve(yte_c[:,i],pp[:,i])
        a=auc(fpr,tpr); aucs[cls]=round(a,4)
        ax.plot(fpr,tpr,color=col,lw=1.5,label=f'{cls} (AUC={a:.3f})')
    ax.plot([0,1],[0,1],'k--',lw=1,label='Random')
    mean_auc=round(np.mean(list(aucs.values())),4)
    ax.set_title(f'{title}\nROC-AUC (Mean={mean_auc})',fontweight='bold')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.legend(fontsize=7,loc='lower right'); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(f'/kaggle/working/{prefix}_roc.png',dpi=150); plt.show()

    return {'tag':title,'train':round(tr_acc*100,2),'val':round(va_acc*100,2),
            'test':round(te_acc*100,2),'mean_auc':mean_auc,'macro_f1':macro_f1}

# ── Model builders ────────────────────────────────────────────────────────────
def build_baseline_mlp(dim):
    inp=keras.Input(shape=(dim,))
    x=layers.Dense(128,activation='relu',kernel_regularizer=regularizers.l2(1e-4))(inp)
    x=layers.Dropout(0.3)(x)
    x=layers.Dense(64, activation='relu',kernel_regularizer=regularizers.l2(1e-4))(x)
    x=layers.Dropout(0.2)(x)
    out=layers.Dense(N_ANN,activation='softmax')(x)
    m=keras.Model(inp,out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy',metrics=['accuracy'])
    return m

def build_improved_mlp(dim):
    inp=keras.Input(shape=(dim,))
    x=layers.Dense(512,activation='relu',kernel_regularizer=regularizers.l2(1e-4))(inp)
    x=layers.BatchNormalization()(x); x=layers.Dropout(0.4)(x)
    x=layers.Dense(256,activation='relu',kernel_regularizer=regularizers.l2(1e-4))(x)
    x=layers.BatchNormalization()(x); x=layers.Dropout(0.3)(x)
    x=layers.Dense(128,activation='relu',kernel_regularizer=regularizers.l2(1e-4))(x)
    x=layers.BatchNormalization()(x); x=layers.Dropout(0.2)(x)
    out=layers.Dense(N_ANN,activation='softmax')(x)
    m=keras.Model(inp,out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy',metrics=['accuracy'])
    return m

print('ANN helpers ready.')

---
## Part 12 — ANN Model 1: Strategy A Baseline (XYZ Only, 63 features)

**Intentionally simple.** Two hidden layers, fixed learning rate, 100 epochs, no callbacks.
Purpose: verify the data pipeline is correct and establish the minimum performance bar before adding complexity.

### Architecture
```
Input(63) → Dense(128, relu) → Dropout(0.3) → Dense(64, relu) → Dropout(0.2) → Dense(14, softmax)
```
**~17,358 trainable parameters**

In [ ]:
os.makedirs('/kaggle/working/strategy_a_xyz/baseline',exist_ok=True)
ann_a_base=build_baseline_mlp(A_tr.shape[1])
ann_a_base.summary()
h_a_base=ann_a_base.fit(A_tr,A_ytr,validation_data=(A_val,A_yval),
    epochs=100,batch_size=32,class_weight=class_weights,verbose=1)
ann_a_base.save('/kaggle/working/strategy_a_xyz/baseline/usl_ann_a_baseline.keras')
r_a_base=ann_evaluate(ann_a_base,A_tr,A_val,A_te,
    A_ytr,A_yval,A_yte,A_ytr_i,A_yval_i,A_yte_i,
    h_a_base,'Strategy A — Baseline MLP (XYZ Only)','ann_a_base')

---
## Part 13 — ANN Model 2: Strategy A Improved (XYZ Only, 63 features)

Same 63-feature input as baseline but with a deeper architecture. Each improvement targets a specific weakness:

| Addition | Weakness Addressed |
|---|---|
| 3 layers (512→256→128) | Baseline too shallow to capture subtle landmark relationships |
| BatchNorm after each Dense | Unstable activations in deep networks |
| ReduceLROnPlateau | Fixed lr caused oscillation in baseline after epoch 50 |
| EarlyStopping | Baseline ran all 100 epochs even after plateau |
| ModelCheckpoint | Ensures peak weights are always saved |

### Architecture
```
Input(63) → Dense(512) → BN → Drop(0.4) → Dense(256) → BN → Drop(0.3)
          → Dense(128) → BN → Drop(0.2) → Dense(14, softmax)
```
**~200,590 trainable parameters**

In [ ]:
os.makedirs('/kaggle/working/strategy_a_xyz/improved',exist_ok=True)
ann_a_imp=build_improved_mlp(A_tr.shape[1])
ann_a_imp.summary()
cb_a_imp=[
    EarlyStopping(monitor='val_accuracy',patience=15,restore_best_weights=True,verbose=1),
    ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=7,min_lr=1e-6,verbose=1),
    ModelCheckpoint('/kaggle/working/strategy_a_xyz/improved/best_a_imp.keras',
                    monitor='val_accuracy',save_best_only=True,verbose=1)]
h_a_imp=ann_a_imp.fit(A_tr,A_ytr,validation_data=(A_val,A_yval),
    epochs=200,batch_size=32,class_weight=class_weights,callbacks=cb_a_imp,verbose=1)
ann_a_imp.save('/kaggle/working/strategy_a_xyz/improved/usl_ann_a_improved.keras')
r_a_imp=ann_evaluate(ann_a_imp,A_tr,A_val,A_te,
    A_ytr,A_yval,A_yte,A_ytr_i,A_yval_i,A_yte_i,
    h_a_imp,'Strategy A — Improved MLP (XYZ Only)','ann_a_imp')

---
## Part 14 — ANN Model 3: Strategy B Baseline (XYZ + Angles, 78 features)

Same shallow baseline architecture but with 78-feature input (63 XYZ + 15 angles). Tests whether angle features alone — without architectural improvements — can boost accuracy.

**Key insight:** Angle features are rotation-invariant and directly encode finger bend (flexion) — the defining characteristic of each USL letter. Strategy B should converge faster and handle similar signs (Gaaf/Zay) better than XYZ-only.

### Architecture
```
Input(78) → Dense(128, relu) → Dropout(0.3) → Dense(64, relu) → Dropout(0.2) → Dense(14, softmax)
```

In [ ]:
os.makedirs('/kaggle/working/strategy_b_xyz_angles/baseline',exist_ok=True)
ann_b_base=build_baseline_mlp(B_tr.shape[1])
ann_b_base.summary()
h_b_base=ann_b_base.fit(B_tr,B_ytr,validation_data=(B_val,B_yval),
    epochs=100,batch_size=32,class_weight=class_weights,verbose=1)
ann_b_base.save('/kaggle/working/strategy_b_xyz_angles/baseline/usl_ann_b_baseline.keras')
r_b_base=ann_evaluate(ann_b_base,B_tr,B_val,B_te,
    B_ytr,B_yval,B_yte,B_ytr_i,B_yval_i,B_yte_i,
    h_b_base,'Strategy B — Baseline MLP (XYZ + Angles)','ann_b_base')

---
## Part 15 — ANN Model 4: Strategy B Improved (XYZ + Angles, 78 features) ★ WINNER

Combines the best of both dimensions: rotation-invariant angle features (Strategy B) with the full 3-layer BatchNorm architecture and adaptive callbacks (Improved). This is the **overall project winner**.

### Architecture
```
Input(78) → Dense(512) → BN → Drop(0.4) → Dense(256) → BN → Drop(0.3)
          → Dense(128) → BN → Drop(0.2) → Dense(14, softmax)
```

**Expected results:** Highest AUC, highest val accuracy, fewest misclassifications — including full resolution of the Gaaf-Zay confusion that persisted in all CNN models.

In [ ]:
os.makedirs('/kaggle/working/strategy_b_xyz_angles/improved',exist_ok=True)
ann_b_imp=build_improved_mlp(B_tr.shape[1])
ann_b_imp.summary()
cb_b_imp=[
    EarlyStopping(monitor='val_accuracy',patience=15,restore_best_weights=True,verbose=1),
    ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=7,min_lr=1e-6,verbose=1),
    ModelCheckpoint('/kaggle/working/strategy_b_xyz_angles/improved/best_b_imp.keras',
                    monitor='val_accuracy',save_best_only=True,verbose=1)]
h_b_imp=ann_b_imp.fit(B_tr,B_ytr,validation_data=(B_val,B_yval),
    epochs=200,batch_size=32,class_weight=class_weights,callbacks=cb_b_imp,verbose=1)
ann_b_imp.save('/kaggle/working/strategy_b_xyz_angles/improved/usl_ann_b_improved.keras')
r_b_imp=ann_evaluate(ann_b_imp,B_tr,B_val,B_te,
    B_ytr,B_yval,B_yte,B_ytr_i,B_yval_i,B_yte_i,
    h_b_imp,'Strategy B — Improved MLP (XYZ + Angles) ★','ann_b_imp')

---
## Part 16 — ANN Four-Model Comparison
> Note: Strategy B Baseline and Strategy A Improved both achieve 98.15% test accuracy. Despite the prediction that angle features would improve convergence, the shallow baseline architecture (2 hidden layers) appears to be the bottleneck — the added geometric information cannot be fully exploited without sufficient model depth. This is resolved in Strategy B Improved, which combines both angle features and the deeper 3-layer architecture to achieve 98.41%.

In [ ]:
ann_results=[r_a_base,r_a_imp,r_b_base,r_b_imp]
ann_df=pd.DataFrame(ann_results)
ann_df.to_csv('/kaggle/working/ann_comparison.csv',index=False)
print('\n'+'='*65+'\n  ANN FOUR-MODEL COMPARISON\n'+'='*65)
print(ann_df.to_string(index=False))

la=['A-Baseline\n(XYZ)','A-Improved\n(XYZ)','B-Baseline\n(XYZ+Ang)','B-Improved\n(XYZ+Ang) ★']
x=np.arange(4); w=0.22
fig,ax=plt.subplots(figsize=(14,6))
b1=ax.bar(x-1.5*w,[r['train'] for r in ann_results],w,label='Train Acc (%)',  color='steelblue',   alpha=0.9)
b2=ax.bar(x-0.5*w,[r['val']   for r in ann_results],w,label='Val Acc (%)',    color='orange',      alpha=0.9)
b3b=ax.bar(x+0.5*w,[r['test']  for r in ann_results],w,label='Test Acc (%)',  color='seagreen',    alpha=0.9)
b4=ax.bar(x+1.5*w,[r['macro_f1']*100 for r in ann_results],w,label='F1×100', color='mediumpurple',alpha=0.9)
for bars in [b1,b2,b3b,b4]: ax.bar_label(bars,fmt='%.1f',padding=2,fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(la,fontsize=10)
ax.set_ylim(0,115); ax.set_title('ANN — Four-Model Comparison',fontsize=13,fontweight='bold')
ax.legend(); ax.grid(axis='y',alpha=0.3)
plt.tight_layout(); plt.savefig('/kaggle/working/ann_4model_cmp.png',dpi=150); plt.show()

# A-Improved vs B-Improved overlay
fig,axes=plt.subplots(1,2,figsize=(13,4))
fig.suptitle('A-Improved vs B-Improved — Val Accuracy & Loss',fontweight='bold')
axes[0].plot(h_a_imp.history['val_accuracy'],label='A-Improved (XYZ)',       color='steelblue',lw=2)
axes[0].plot(h_b_imp.history['val_accuracy'],label='B-Improved (XYZ+Angles)',color='orange',   lw=2)
axes[0].set_title('Val Accuracy'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(h_a_imp.history['val_loss'],label='A-Improved',color='steelblue',lw=2)
axes[1].plot(h_b_imp.history['val_loss'],label='B-Improved',color='orange',   lw=2)
axes[1].set_title('Val Loss'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig('/kaggle/working/ann_a_vs_b_overlay.png',dpi=150); plt.show()

If you look at the raw numbers, **Strategy B – Improved MLP (XYZ + Angles)** cleanly takes the crown, leading in both **test accuracy** (**98.41%** vs. **98.15%** for Strategy A Improved) and **macro F1** (**0.9835** vs. **0.9809**).

Here is the corrected breakdown matching your exact table metrics:

### 1. Robustness & Geometric Invariance (The "Angles" Advantage)

Relying solely on raw XYZ spatial coordinates (Strategy A) leaves a model highly sensitive to camera distance (scaling), shifting positions (translation), and slight hand tilts (rotation). By explicitly training the model on **Angles** alongside XYZ coordinates, Strategy B forces the network to learn the structural, semantic relationships between points. This geometric awareness is why Strategy B stands out as the most reliable choice when facing real-world orientation changes and varied user setups.

### 2. Superior Generalization (Taming Overfitting)

Let's look at the delta between training performance and testing performance across the models:

* **Strategy A (Baseline):** **99.80%** (Train) $\rightarrow$ **97.88%** (Test) | **Drop:** **1.92%**
* **Strategy A (Improved):** **99.40%** (Train) $\rightarrow$ **98.15%** (Test) | **Drop:** **1.25%**
* **Strategy B (Improved):** **99.60%** (Train) $\rightarrow$ **98.41%** (Test) | **Drop:** **1.19%**

> **Note:** Strategy B (Improved) delivers the highest test performance of the bunch while maintaining the smallest performance gap (**1.19%**) between training and testing. This tight variance indicates a highly stable, well-regularized model that isn't just memorizing the coordinates of the dataset, making it much more resilient against out-of-distribution data.

### 3. Peak Class Discrimination

Strategy B (Improved) ties with Strategy A (Improved) for the highest `mean_auc` at **0.9997**. This means its overall ability to distinguish between distinct classes across various decision thresholds is exceptionally high, giving you cleaner confidence scores and fewer chaotic misclassifications during live inference.

---
## Part 17 — ANN Winner Variant A: Higher LR + L1L2 Regularisation

**Hyperparameter changes from the original B-Improved:**

| Parameter | Original B-Improved | Variant A |
|---|---|---|
| Learning Rate | Adam lr=1e-3 | **Adam lr=5e-3 (5× higher)** |
| Regularisation | L2=1e-4 | **L1L2 (l1=1e-5, l2=1e-4)** |
| Batch Size | 32 | **64** — larger batches for gradient stability |
| ReduceLR patience | 7 | **5** — faster reaction to plateau |

**Why L1L2?** L1 regularisation promotes sparsity — some landmark features may be redundant and L1 encourages the model to zero them out. Combined with L2 (elastic net), this can produce a more focused, generalizable model.
**Why higher LR?** Larger initial LR can escape local minima faster; ReduceLROnPlateau will correct it downward automatically.

In [ ]:
os.makedirs('/kaggle/working/ann_b_variant_a',exist_ok=True)

inp=keras.Input(shape=(B_tr.shape[1],))
x=layers.Dense(512,activation='relu',
               kernel_regularizer=regularizers.L1L2(l1=1e-5,l2=1e-4))(inp)
x=layers.BatchNormalization()(x); x=layers.Dropout(0.4)(x)
x=layers.Dense(256,activation='relu',
               kernel_regularizer=regularizers.L1L2(l1=1e-5,l2=1e-4))(x)
x=layers.BatchNormalization()(x); x=layers.Dropout(0.3)(x)
x=layers.Dense(128,activation='relu',
               kernel_regularizer=regularizers.L1L2(l1=1e-5,l2=1e-4))(x)
x=layers.BatchNormalization()(x); x=layers.Dropout(0.2)(x)
out=layers.Dense(N_ANN,activation='softmax')(x)
ann_bv1=keras.Model(inp,out)
ann_bv1.compile(optimizer=keras.optimizers.Adam(5e-3),
                loss='categorical_crossentropy',metrics=['accuracy'])
ann_bv1.summary()

cb_bv1=[
    EarlyStopping(monitor='val_accuracy',patience=15,restore_best_weights=True,verbose=1),
    ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=5,min_lr=1e-6,verbose=1),
    ModelCheckpoint('/kaggle/working/ann_b_variant_a/best_bv1.keras',
                    monitor='val_accuracy',save_best_only=True,verbose=1)]
print('ANN B-Improved Variant A — Higher LR + L1L2')
h_bv1=ann_bv1.fit(B_tr,B_ytr,validation_data=(B_val,B_yval),
    epochs=200,batch_size=64,class_weight=class_weights,callbacks=cb_bv1,verbose=1)
ann_bv1.save('/kaggle/working/ann_b_variant_a/usl_ann_b_variant_a.keras')
r_bv1=ann_evaluate(ann_bv1,B_tr,B_val,B_te,
    B_ytr,B_yval,B_yte,B_ytr_i,B_yval_i,B_yte_i,
    h_bv1,'ANN B-Improved Variant A (Higher LR + L1L2)','ann_bv1')

---
## Part 18 — ANN Winner Variant B: Heavier Dropout + Cosine LR Decay

**Hyperparameter changes from the original B-Improved:**

| Parameter | Original B-Improved | Variant B |
|---|---|---|
| Dropout rates | 0.4, 0.3, 0.2 | **0.5, 0.4, 0.3** — heavier regularisation |
| LR Schedule | ReduceLROnPlateau | **Cosine Annealing** |
| Layer width | 512→256→128 | **256→128→64** — narrower, more compact |
| Optimizer | Adam lr=1e-3 | **Adam lr=2e-3** |

**Why heavier dropout?** The original model may still slightly overfit in later epochs. Increasing dropout rates pushes the model to rely on more distributed feature representations.
**Why cosine + narrower?** A narrower network with cosine annealing tests whether a more compact representation with smooth LR decay can match the original — useful for understanding the minimum model complexity needed.

In [ ]:
os.makedirs('/kaggle/working/ann_b_variant_b',exist_ok=True)

total_steps_ann = 200 * (len(B_tr)//32)
cosine_ann = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=2e-3, decay_steps=total_steps_ann, alpha=1e-6)

inp=keras.Input(shape=(B_tr.shape[1],))
x=layers.Dense(256,activation='relu',kernel_regularizer=regularizers.l2(1e-4))(inp)
x=layers.BatchNormalization()(x); x=layers.Dropout(0.5)(x)
x=layers.Dense(128,activation='relu',kernel_regularizer=regularizers.l2(1e-4))(x)
x=layers.BatchNormalization()(x); x=layers.Dropout(0.4)(x)
x=layers.Dense(64, activation='relu',kernel_regularizer=regularizers.l2(1e-4))(x)
x=layers.BatchNormalization()(x); x=layers.Dropout(0.3)(x)
out=layers.Dense(N_ANN,activation='softmax')(x)
ann_bv2=keras.Model(inp,out)
ann_bv2.compile(optimizer=keras.optimizers.Adam(cosine_ann),
                loss='categorical_crossentropy',metrics=['accuracy'])
ann_bv2.summary()

cb_bv2=[
    EarlyStopping(monitor='val_accuracy',patience=20,restore_best_weights=True,verbose=1),
    ModelCheckpoint('/kaggle/working/ann_b_variant_b/best_bv2.keras',
                    monitor='val_accuracy',save_best_only=True,verbose=1)]
print('ANN B-Improved Variant B — Heavier Dropout + Cosine LR')
h_bv2=ann_bv2.fit(B_tr,B_ytr,validation_data=(B_val,B_yval),
    epochs=200,batch_size=32,class_weight=class_weights,callbacks=cb_bv2,verbose=1)
ann_bv2.save('/kaggle/working/ann_b_variant_b/usl_ann_b_variant_b.keras')
r_bv2=ann_evaluate(ann_bv2,B_tr,B_val,B_te,
    B_ytr,B_yval,B_yte,B_ytr_i,B_yval_i,B_yte_i,
    h_bv2,'ANN B-Improved Variant B (Heavier Dropout + Cosine)','ann_bv2')

---
## Part 19 — ANN Winner Variants Comparison

In [ ]:
ann_variants=[r_b_imp,r_bv1,r_bv2]
annv_df=pd.DataFrame(ann_variants)
annv_df.to_csv('/kaggle/working/ann_b_variants_comparison.csv',index=False)
print(annv_df.to_string(index=False))

vnames_ann=['Original\n(Adam+ReduceLR)','Variant A\n(HigherLR+L1L2)','Variant B\n(HeavyDrop+Cosine)']
x=np.arange(3); w=0.25
fig,ax=plt.subplots(figsize=(12,5))
b1=ax.bar(x-w,[r['test']      for r in ann_variants],w,label='Test Acc (%)', color='steelblue',   alpha=0.9)
b2=ax.bar(x,  [r['mean_auc']*100 for r in ann_variants],w,label='AUC×100',  color='orange',      alpha=0.9)
b3b=ax.bar(x+w,[r['macro_f1']*100 for r in ann_variants],w,label='F1×100', color='seagreen',    alpha=0.9)
for bars in [b1,b2,b3b]: ax.bar_label(bars,fmt='%.1f',padding=2,fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(vnames_ann,fontsize=10)
ax.set_ylim(0,115)
ax.set_title('ANN B-Improved — Original vs Variant A vs Variant B',fontsize=13,fontweight='bold')
ax.legend(); ax.grid(axis='y',alpha=0.3)
plt.tight_layout(); plt.savefig('/kaggle/working/ann_variants_cmp.png',dpi=150); plt.show()

# Val accuracy overlay
fig,ax=plt.subplots(figsize=(11,4))
ax.plot(h_b_imp.history['val_accuracy'],label='Original',  color='steelblue',lw=2)
ax.plot(h_bv1.history['val_accuracy'],  label='Variant A', color='orange',   lw=2)
ax.plot(h_bv2.history['val_accuracy'],  label='Variant B', color='seagreen', lw=2)
ax.set_title('ANN B-Improved Variants — Val Accuracy Overlay',fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val Accuracy')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('/kaggle/working/ann_variants_overlay.png',dpi=150); plt.show()

---
## Part 20 — Overall Project Winner: CNN vs ANN


In [ ]:
all_models=[
    {'Model':'CNN EfficientNetB0 (128×128)', 'Type':'CNN',**{k:v for k,v in r_b0.items()  if k!='model'}},
    {'Model':'CNN EfficientNetB3 (224×224)', 'Type':'CNN',**{k:v for k,v in r_b3.items()  if k!='model'}},
    {'Model':'CNN ResNet50 (224×224)',       'Type':'CNN',**{k:v for k,v in r_rn.items()  if k!='model'}},
    {'Model':'CNN MobileNetV2 (128×128)',    'Type':'CNN',**{k:v for k,v in r_mv2.items() if k!='model'}},
    {'Model':'CNN RN Variant A (SGD)',       'Type':'CNN',**{k:v for k,v in r_rnv1.items() if k!='model'}},
    {'Model':'CNN RN Variant B (RMSprop)',   'Type':'CNN',**{k:v for k,v in r_rnv2.items() if k!='model'}},
    {'Model':'ANN A-Baseline (XYZ)',         'Type':'ANN','test_acc':r_a_base['test'],'mean_auc':r_a_base['mean_auc'],'macro_f1':r_a_base['macro_f1']},
    {'Model':'ANN A-Improved (XYZ)',         'Type':'ANN','test_acc':r_a_imp['test'], 'mean_auc':r_a_imp['mean_auc'], 'macro_f1':r_a_imp['macro_f1']},
    {'Model':'ANN B-Baseline (XYZ+Angles)',  'Type':'ANN','test_acc':r_b_base['test'],'mean_auc':r_b_base['mean_auc'],'macro_f1':r_b_base['macro_f1']},
    {'Model':'ANN B-Improved (XYZ+Angles) ★','Type':'ANN','test_acc':r_b_imp['test'], 'mean_auc':r_b_imp['mean_auc'], 'macro_f1':r_b_imp['macro_f1']},
    {'Model':'ANN B Variant A (HighLR+L1L2)','Type':'ANN','test_acc':r_bv1['test'],   'mean_auc':r_bv1['mean_auc'],  'macro_f1':r_bv1['macro_f1']},
    {'Model':'ANN B Variant B (Cosine+Drop)','Type':'ANN','test_acc':r_bv2['test'],   'mean_auc':r_bv2['mean_auc'],  'macro_f1':r_bv2['macro_f1']},
]
all_df=pd.DataFrame(all_models)
all_df.to_csv('/kaggle/working/complete_comparison.csv',index=False)
print('\n'+'='*70+'\n  COMPLETE PROJECT COMPARISON\n'+'='*70)
print(all_df[['Model','Type','test_acc','mean_auc','macro_f1']].to_string(index=False))

# Bar chart
c_colors=['steelblue']*6+['coral']*6
fig,ax=plt.subplots(figsize=(18,6))
bars=ax.bar(all_df['Model'],all_df['test_acc'],color=c_colors,edgecolor='white',linewidth=0.8)
ax.bar_label(bars,fmt='%.2f%%',padding=3,fontsize=7.5)
ax.set_ylim(80,105); ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Complete Project — All Models Test Accuracy',fontsize=13,fontweight='bold')
plt.xticks(rotation=25,ha='right',fontsize=8)
from matplotlib.patches import Patch
leg=[Patch(color='steelblue',label='CNN Models'),Patch(color='coral',label='ANN Models')]
ax.legend(handles=leg); ax.grid(axis='y',alpha=0.3)
plt.tight_layout(); plt.savefig('/kaggle/working/complete_comparison.png',dpi=150); plt.show()

---
## Part 21 — Random Image Testing

Both winners are tested on random unseen images:
- **CNN Winner:** RESNET 50 (224×224) — MediaPipe crop → preprocess → predict
- **ANN Winner:** Strategy B Improved (XYZ+Angles) — MediaPipe landmarks → wrist-normalise → angle features → scale → predict

### ⚙️ Set Your Test Images Path Below
Update `RANDOM_IMAGES_DIR` to point to your test images folder. Images can be in any of these formats: `.jpg`, `.jpeg`, `.png`, `.bmp`.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  SET YOUR RANDOM IMAGES PATH HERE                           ║
# ╚══════════════════════════════════════════════════════════════╝
#you may also use /kaggle/input/datasets/assamirzafar/random-images
RANDOM_IMAGES_DIR = '/kaggle/input/datasets/umaimak/random-images-1'
N_SHOW = 20  # number of images to show in each visual grid

VALID_EXT = {'.jpg','.jpeg','.png','.bmp','.webp'}
all_test_imgs = sorted([
    os.path.join(RANDOM_IMAGES_DIR,f)
    for f in os.listdir(RANDOM_IMAGES_DIR)
    if os.path.splitext(f)[1].lower() in VALID_EXT
])
print(f'Found {len(all_test_imgs)} test images in: {RANDOM_IMAGES_DIR}')

### 21.1 — CNN Prediction Pipeline (RESNET50)

In [ ]:
def cnn_predict_one(img_path, model, size, preprocess_fn):
    img_bgr = cv2.imread(img_path)
    if img_bgr is None: return None, None, False
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_bgr.shape[:2]; annotated = img_bgr.copy()
    with mp_hands.Hands(static_image_mode=True, max_num_hands=1,
                        min_detection_confidence=CONF) as hands:
        res = hands.process(img_rgb)
    found = False
    if res.multi_hand_landmarks:
        found = True
        mp_drawing.draw_landmarks(annotated, res.multi_hand_landmarks[0],
            mp_hands.HAND_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(0,255,0), thickness=2, circle_radius=3),
            mp_drawing.DrawingSpec(color=(255,0,0), thickness=2))
        lms = res.multi_hand_landmarks[0].landmark
        xs = [l.x*w for l in lms]; ys = [l.y*h for l in lms]
        x1 = max(0, int(min(xs))-PADDING); y1 = max(0, int(min(ys))-PADDING)
        x2 = min(w, int(max(xs))+PADDING); y2 = min(h, int(max(ys))+PADDING)
        crop = img_bgr[y1:y2, x1:x2]
    else:
        crop = img_bgr
    rgb = cv2.cvtColor(cv2.resize(crop, (size, size)), cv2.COLOR_BGR2RGB)
    arr = preprocess_fn(rgb.astype(np.float32))[np.newaxis]
    probs = model.predict(arr, verbose=0)[0]
    top3 = [(CLASS_NAMES[i], round(float(probs[i])*100, 2))
          for i in np.argsort(probs)[::-1][:3]]
    return top3, annotated, found

### 21.2 — ANN Prediction Pipeline (Strategy B Improved)

In [ ]:
def ann_predict_one(img_path,model,scaler):
    img_bgr=cv2.imread(img_path)
    if img_bgr is None: return None,None,False
    img_rgb=cv2.cvtColor(img_bgr,cv2.COLOR_BGR2RGB)
    annotated=img_bgr.copy()
    with mp_hands.Hands(static_image_mode=True,max_num_hands=1,
                        min_detection_confidence=CONF) as hands:
        res=hands.process(img_rgb)
    if not res.multi_hand_landmarks: return None,img_bgr,False
    mp_drawing.draw_landmarks(annotated,res.multi_hand_landmarks[0],
        mp_hands.HAND_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(0,255,0),thickness=2,circle_radius=3),
        mp_drawing.DrawingSpec(color=(255,0,0),thickness=2))
    c=[]
    for l in res.multi_hand_landmarks[0].landmark: c.extend([l.x,l.y,l.z])
    c[0::3]=[v-c[0] for v in c[0::3]]
    c[1::3]=[v-c[1] for v in c[1::3]]
    c[2::3]=[v-c[2] for v in c[2::3]]
    xyz=np.array(c,dtype=np.float32).reshape(1,-1)
    ang=extract_angles(xyz)
    feat=scaler.transform(np.concatenate([xyz,ang],axis=1))
    probs=model.predict(feat,verbose=0)[0]
    top3=[(ANN_CLASSES[i],round(float(probs[i])*100,2))
          for i in np.argsort(probs)[::-1][:3]]
    return top3,annotated,True

### 21.3 — Run Both Models on All Test Images

In [ ]:
cnn_preds=[]; ann_preds=[]
print('Running predictions on all test images...')
print(f'{"Image":40s}  {"CNN-RN":20s}  {"ANN-B-Imp":20s}')
print('-'*85)
for path in all_test_imgs:
    ct,ca,cf=cnn_predict_one(path,rn,224,rn_pre)
    at,aa,af=ann_predict_one(path,ann_b_imp,sc_b)
    cnn_preds.append({'path':path,'top3':ct,'annotated':ca,'found':cf})
    ann_preds.append({'path':path,'top3':at,'annotated':aa,'found':af})
    cn=f"{ct[0][0]} ({ct[0][1]:.1f}%)" if ct else 'NO HAND'
    an=f"{at[0][0]} ({at[0][1]:.1f}%)" if at else 'NO HAND'
    print(f'{os.path.basename(path):40s}  {cn:20s}  {an}')

### 21.4 — Visual Grid: CNN Winner (RESNET 50)

In [ ]:
def draw_grid(preds,title,save_path,n_show=N_SHOW):
    valid=[p for p in preds if p['top3'] is not None]
    sample=random.sample(valid,min(n_show,len(valid)))
    cols=5; rows_n=(len(sample)+cols-1)//cols
    fig,axes=plt.subplots(rows_n,cols,figsize=(cols*3.5,rows_n*3.8))
    axes=axes.flatten()
    for ax in axes: ax.axis('off')
    for i,p in enumerate(sample):
        ax=axes[i]
        img=cv2.cvtColor(p['annotated'],cv2.COLOR_BGR2RGB)
        conf=p['top3'][0][1]; pred=p['top3'][0][0]
        col='#27ae60' if conf>=90 else ('#f39c12' if conf>=70 else '#e74c3c')
        ax.imshow(img)
        ax.set_title(f'{pred}\n{conf:.1f}%',fontsize=10,fontweight='bold',color=col,pad=4)
        # Show top-3 below
        t3='  '.join([f"{c}:{v:.0f}%" for c,v in p['top3'][1:3]])
        ax.set_xlabel(t3,fontsize=7,color='grey')
        for spine in ax.spines.values():
            spine.set_edgecolor(col); spine.set_linewidth(3); spine.set_visible(True)
    patches=[mpatches.Patch(color='#27ae60',label='≥90% conf'),
             mpatches.Patch(color='#f39c12',label='70–90%'),
             mpatches.Patch(color='#e74c3c',label='<70%')]
    fig.legend(handles=patches,loc='lower center',ncol=3,fontsize=10,bbox_to_anchor=(0.5,-0.02))
    fig.suptitle(title,fontsize=14,fontweight='bold',y=1.01)
    plt.tight_layout()
    plt.savefig(save_path,dpi=150,bbox_inches='tight'); plt.show()
    print(f'Saved → {save_path}')

# FIXED: Title and save destination updated to reflect ResNet50
draw_grid(cnn_preds,'CNN Winner — ResNet50 (224×224)',
          '/kaggle/working/test_cnn_rn_grid.png')

### 21.5 — Visual Grid: ANN Winner (Strategy B Improved)

In [ ]:
draw_grid(ann_preds,'ANN Winner — Strategy B Improved (XYZ + Angles)',
          '/kaggle/working/test_ann_b_imp_grid.png')

### 21.6 — Agreement / Disagreement Analysis

In [ ]:
rows=[]
for c_p,a_p in zip(cnn_preds,ann_preds):
    cp=c_p['top3'][0][0] if c_p['top3'] else 'NO HAND'
    ap=a_p['top3'][0][0] if a_p['top3'] else 'NO HAND'
    rows.append({'image':os.path.basename(c_p['path']),
                 'cnn_pred':cp,'cnn_conf':c_p['top3'][0][1] if c_p['top3'] else 0,
                 'ann_pred':ap,'ann_conf':a_p['top3'][0][1] if a_p['top3'] else 0,
                 'agree':cp==ap})
ag_df=pd.DataFrame(rows)
ag_df.to_csv('/kaggle/working/cnn_vs_ann_agreement.csv',index=False)

total=len(ag_df); agreed=ag_df['agree'].sum()
# FIXED: Label changed from CNN-B3 to CNN-RN
print(f'CNN-RN vs ANN-B-Improved Agreement')
print(f'  Total   : {total}')
print(f'  Agreed  : {agreed} ({agreed/total*100:.1f}%)')
print(f'  Disagree: {total-agreed} ({(total-agreed)/total*100:.1f}%)')
dis=ag_df[~ag_df['agree']]
if len(dis)>0:
    print('\nDisagreements:')
    print(dis[['image','cnn_pred','cnn_conf','ann_pred','ann_conf']].to_string(index=False))
else:
    print('\nBoth models agreed on every test image!')

In [ ]:
# Disagreement side-by-side grid
dis_idx=[i for i,r in enumerate(rows) if not r['agree']]
if not dis_idx:
    print('No disagreements to show.')
else:
    n_dis=len(dis_idx)
    fig,axes=plt.subplots(n_dis,2,figsize=(10,n_dis*3.5))
    if n_dis==1: axes=axes[np.newaxis,:]
    
    # FIXED: Updated title from CNN B3 to CNN ResNet50
    fig.suptitle('Disagreements — CNN ResNet50 (left) vs ANN B-Improved (right)',
                 fontsize=13,fontweight='bold',y=1.01)
    for row,idx in enumerate(dis_idx):
        for col,(pred_rec,tag) in enumerate([
                (cnn_preds[idx],'CNN ResNet50'), # FIXED: Updated legend tag label
                (ann_preds[idx],'ANN B-Improved')]):
            ax=axes[row,col]
            img=cv2.cvtColor(pred_rec['annotated'],cv2.COLOR_BGR2RGB)
            conf=pred_rec['top3'][0][1] if pred_rec['top3'] else 0
            pred=pred_rec['top3'][0][0] if pred_rec['top3'] else 'N/A'
            color='#27ae60' if conf>=90 else ('#f39c12' if conf>=70 else '#e74c3c')
            ax.imshow(img)
            ax.set_title(f'{tag}\n{pred} ({conf:.1f}%)',
                         fontsize=10,fontweight='bold',color=color)
            ax.set_xlabel(os.path.basename(cnn_preds[idx]['path']),fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_edgecolor(color); spine.set_linewidth(3); spine.set_visible(True)
    plt.tight_layout()
    plt.savefig('/kaggle/working/disagreements_grid.png',dpi=150,bbox_inches='tight')
    plt.show()
    print(f'Saved → disagreements_grid.png ({n_dis} disagreements)')

In [ ]:
# Confidence distribution comparison
c_valid=ag_df[ag_df['cnn_conf']>0]['cnn_conf']
a_valid=ag_df[ag_df['ann_conf']>0]['ann_conf']
fig,axes=plt.subplots(1,2,figsize=(14,5))

# FIXED: Updated main title to reflect ResNet50
fig.suptitle('CNN-RN vs ANN-B-Improved — Confidence Analysis',fontweight='bold')

# FIXED: Label updated from CNN-B3 to CNN-RN
axes[0].hist(c_valid,bins=20,alpha=0.7,color='steelblue',label='CNN-RN',edgecolor='white')
axes[0].hist(a_valid,bins=20,alpha=0.7,color='coral',    label='ANN-B-Imp',edgecolor='white')
axes[0].axvline(90,color='green',linestyle='--',lw=1.5,label='90% line')
axes[0].set_title('Confidence Distribution'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_xlabel('Confidence (%)'); axes[0].set_ylabel('Count')

cls_agree={}
for cls in CLASS_NAMES:
    sub=ag_df[ag_df['cnn_pred']==cls]
    if len(sub)>0: cls_agree[cls]=sub['agree'].mean()*100
if cls_agree:
    cls_n=list(cls_agree.keys()); cls_v=list(cls_agree.values())
    cols_a=['#27ae60' if v==100 else '#f39c12' if v>=70 else '#e74c3c' for v in cls_v]
    axes[1].bar(cls_n,cls_v,color=cols_a,edgecolor='white')
    axes[1].set_ylim(0,115); axes[1].set_ylabel('Agreement %')
    axes[1].set_title('Per-Class Agreement Rate (by CNN prediction)')
    axes[1].tick_params(axis='x',rotation=35); axes[1].grid(axis='y',alpha=0.3)
    ax2=axes[1]; ax2.bar_label(ax2.containers[0],fmt='%.0f%%',padding=2,fontsize=8)
plt.tight_layout()
plt.savefig('/kaggle/working/confidence_agreement.png',dpi=150); plt.show()

## Final Project Summary

| Model | Type | Input | Test Acc | AUC | F1 |
| --- | --- | --- | --- | --- | --- |
| EfficientNetB0 | CNN | $128 \times 128$ | 89.78% | 0.9944 | 0.8967 |
| EfficientNetB3 | CNN | $224 \times 224$ | 90.52% | 0.9946 | 0.9021 |
| ResNet50 | CNN | $224 \times 224$ | 99.00% | 0.9999 | 0.9896 |
| MobileNetV2 | CNN | $128 \times 128$ | 94.51% | 0.9980 | 0.9433 |
| CNN RN Variant A (SGD) | CNN | $224 \times 224$ | 94.01% | 0.9982 | 0.9376 |
| CNN RN Variant B (RMSprop) | CNN | $224 \times 224$ | 94.01% | 0.9978 | 0.9383 |
| ANN A-Baseline (XYZ) | ANN | 63 lm | 97.88% | 0.9996 | 0.9782 |
| ANN A-Improved (XYZ) | ANN | 63 lm | 98.15% | 0.9997 | 0.9809 |
| ANN B-Baseline (XYZ+Ang) | ANN | 78 lm | 98.15% | 0.9996 | 0.9808 |
| ANN B-Improved (XYZ+Ang) | ANN | 78 lm | 98.41% | 0.9997 | 0.9835 |
| ANN B Variant A (L1L2) | ANN | 78 lm | 98.41% | 0.9994 | 0.9833 |
| ANN B Variant B (Cosine) | ANN | 78 lm | 98.41% | 0.9991 | 0.9837 |

---

### Generative AI Use Declaration

* **Claude AI** — Debugging, preprocessing scripts, code structure, report writing assistance
* **Gemini AI** — Research, planning, and idea generation

*All final implementation, model training, analysis, and conclusions were performed by the student group.*